# Module 2 v2 — 3-class masks for ROI (background / strawberry / peduncle)

Builds training targets for the multi-class U-Net (Module 3 v2).

- **Class 0:** background  
- **Class 1:** strawberry (fruit) — from YOLO polygon or HSV fallback  
- **Class 2:** peduncle / crown (always drawn from annotations; overwrites 1)  

Uses the **full-resolution images in `peduncle_masks/images/`** (1008×756), runs the trained YOLOv11 model to get the same ROI crop as Module 1, then crops both RGB and label map. This avoids name mismatches between `roi_crops/` and Roboflow export names.

Output: `multiclass_masks/images/` (RGB ROI crops) and `multiclass_masks/masks_3class/` (single-channel uint8, values 0/1/2).

## Cell 1 — Imports and setup

Loads all required libraries, defines project paths and output directories, sets asymmetric padding constants (`PAD_TOP_FRAC=1.0`, `PAD_SIDE_FRAC=0.25`, `PAD_BOT=20`), detection confidence threshold (`CONF=0.15`), minimum content filter (`MIN_CONTENT=200` non-background pixels), clears output folders, configures the four contributor label sources, and loads the YOLOv11 weights from Module 1.

**Expected output:** confirms the YOLO weights file exists and that CUDA is available.

In [ ]:
%matplotlib inline
import os, glob, re
import numpy as np
import cv2
import torch
from ultralytics import YOLO

PROJECT = r"C:\Users\markm\Documents\RDP\AIDA 2158\Final Project"
PMASK_IMGS = os.path.join(PROJECT, "peduncle_masks", "images")
OUT_IMGS  = os.path.join(PROJECT, "multiclass_masks", "images")
OUT_LBL   = os.path.join(PROJECT, "multiclass_masks", "masks_3class")
_cand = [os.path.join(PROJECT, "runs", "strawberry_seg", "weights", "best.pt"),
         os.path.join(PROJECT, "runs", "segment", "strawberry_seg", "weights", "best.pt")]
YOLO_W = next((p for p in _cand if os.path.exists(p)), _cand[0])
PAD_TOP_FRAC  = 1.00   # extend top by 100% of bbox height  (captures full peduncle)
PAD_SIDE_FRAC = 0.25   # extend each side by 25% of bbox width (angled peduncles)
PAD_BOT       = 20     # small fixed bottom pad (peduncle is above the fruit)
CONF = 0.15            # raised from 0.01 — filters YOLO noise/false-positives
MIN_CONTENT = 200      # minimum non-background pixels in a crop mask; skip pure-BG crops
# Clear output dirs so stale single-crop files don't mix with new per-fruit crops
import shutil
for _d in [OUT_IMGS, OUT_LBL]:
    if os.path.exists(_d): shutil.rmtree(_d)
    os.makedirs(_d)

SOURCES = [
    ("hervejunior", os.path.join(PROJECT, "peduncle.yolov11", "train", "labels"), 0, 1),
    ("biberdork",  os.path.join(PROJECT, "Strawberry Peduncles.yolov11", "train", "labels"), 0, 1),
    ("aida2154",   os.path.join(PROJECT, "400-500 strawberry images.yolov11", "train", "labels"), 0, 1),
    ("markm",      os.path.join(PROJECT, "AIDA 2158 Final.yolov11", "train", "labels"), 0, 1),
]
PED_CLS, FRU_CLS = 0, 1  # as in all Roboflow exports
device = 0 if torch.cuda.is_available() else 'cpu'
print("YOLO weights:", YOLO_W, "exists:", os.path.exists(YOLO_W))
if not os.path.exists(YOLO_W):
    raise FileNotFoundError(f"Missing YOLO weights: {YOLO_W} — run Module 1 first.")
model = YOLO(YOLO_W)
print(f"Loaded YOLO. CUDA: {torch.cuda.is_available()}")

## Cell 2 — Helper functions

Defines the five functions used by the mask generation loop:
- `find_label_path` — maps a `peduncle_masks` filename to its contributor label file.
- `yolo_line_to_full_mask` — converts one YOLO label line (polygon or bbox) to a binary mask.
- `build_multiclass_from_labels` — builds the full (H×W) 3-class map from all label lines.
- `hsv_strawberry_refine` — fills red (HSV) regions as class 1 where no annotation exists.
- `yolo_all_boxes` — runs YOLOv11 on a full image and returns **one asymmetrically-padded crop box per detected strawberry** (top +100% bbox height, sides +25% bbox width, bottom 20px fixed).

In [ ]:
def find_label_path(img_basename: str):
    """Map peduncle_masks filename like 'hervejunior_xxx' -> (labels_dir, stem_without_prefix)."""
    for prefix, lbl_dir, _p, _f in SOURCES:
        pfx = prefix + "_"
        if img_basename.startswith(pfx):
            stem = img_basename[len(pfx) :]
            p = os.path.join(lbl_dir, stem + ".txt")
            if os.path.exists(p):
                return lbl_dir, stem, p
    return None


def yolo_line_to_full_mask(h: int, w: int, line: str, cls: int) -> np.ndarray | None:
    """One line -> binary uint8 {0,1} for that class region."""
    parts = line.strip().split()
    if len(parts) < 2:
        return None
    ci = int(parts[0])
    if ci != cls:
        return None
    vals = list(map(float, parts[1:]))
    m = np.zeros((h, w), dtype=np.uint8)
    if len(vals) == 4:
        cx, cy, bw, bh = vals
        x1 = int((cx - bw / 2) * w); y1 = int((cy - bh / 2) * h)
        x2 = int((cx + bw / 2) * w); y2 = int((cy + bh / 2) * h)
        x1, y1 = max(0, x1), max(0, y1); x2, y2 = min(w, x2), min(h, y2)
        cv2.rectangle(m, (x1, y1), (x2, y2), 1, -1)
        return m
    if len(vals) < 6:
        return None
    pts = []
    for i in range(0, len(vals) - 1, 2):
        px = int(vals[i] * w); py = int(vals[i + 1] * h)
        pts.append([px, py])
    if len(pts) < 3:
        return None
    poly = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
    cv2.fillPoly(m, [poly], 1)
    return m


def build_multiclass_from_labels(lbl_path: str, h: int, w: int) -> np.ndarray:
    """(H,W) uint8 with values 0,1,2."""
    out = np.zeros((h, w), dtype=np.uint8)
    if not os.path.exists(lbl_path):
        return out
    lines = open(lbl_path).read().strip().splitlines()
    # pass 1: strawberry (1)
    for line in lines:
        m = yolo_line_to_full_mask(h, w, line, FRU_CLS)
        if m is not None:
            out[m > 0] = 1
    # pass 2: peduncle (2) overwrites
    for line in lines:
        m = yolo_line_to_full_mask(h, w, line, PED_CLS)
        if m is not None:
            out[m > 0] = 2
    return out


def hsv_strawberry_refine(bgr: np.ndarray, base: np.ndarray) -> None:
    """In-place: set class 1 for red HSV where base==0 (not already fruit/ped)."""
    h, w = base.shape
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    m1 = cv2.inRange(hsv, (0, 40, 40), (15, 255, 255))
    m2 = cv2.inRange(hsv, (165, 40, 40), (180, 255, 255))
    red = cv2.bitwise_or(m1, m2)
    fill = (red > 0) & (base == 0)
    base[fill] = 1


def yolo_all_boxes(img_path: str):
    """Returns list of asymmetrically-padded (x1,y1,x2,y2) for every detected strawberry.
    Top: +100% of bbox height (captures full peduncle above fruit).
    Left/Right: +25% of bbox width (captures angled peduncles).
    Bottom: fixed PAD_BOT px (peduncle is above, not below).
    """
    bgr0 = cv2.imread(img_path)
    if bgr0 is None:
        return []
    H, W = bgr0.shape[:2]
    r = model.predict(img_path, imgsz=640, device=device, conf=CONF, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return []
    result = []
    for box in r.boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = box.astype(int)
        bh = max(1, y2 - y1); bw = max(1, x2 - x1)
        pad_top  = int(bh * PAD_TOP_FRAC)
        pad_side = int(bw * PAD_SIDE_FRAC)
        x1 = max(0, x1 - pad_side); y1 = max(0, y1 - pad_top)
        x2 = min(W, x2 + pad_side); y2 = min(H, y2 + PAD_BOT)
        result.append((int(x1), int(y1), int(x2), int(y2)))
    return result

print("Helper functions defined: find_label_path, yolo_line_to_full_mask, "
      "build_multiclass_from_labels, hsv_strawberry_refine, yolo_all_boxes")

## Cell 3 — Generate 3-class masks for all images

For each image in `peduncle_masks/images/`:
1. Looks up the matching YOLO label file (any of the 4 contributor exports).
2. Builds a 3-class label map (0 = bg, 1 = fruit, 2 = peduncle).
3. Applies HSV-based strawberry fill where no annotation exists.
4. Re-applies peduncle (class 2) on top to prevent it being overwritten.
5. Runs YOLOv11 and gets **one crop per detected strawberry** (not just the largest). Each crop uses asymmetric padding: top +100% bbox height (to capture the peduncle above), sides +25% bbox width (angled peduncles), bottom 20px.
6. Saves each crop as `base_00.png`, `base_01.png`, etc.:
   - RGB crop → `multiclass_masks/images/`
   - 3-class mask → `multiclass_masks/masks_3class/`

**Note:** Output folders are cleared at startup to avoid mixing old single-crop files with new per-fruit crops.

In [ ]:
all_png = sorted(glob.glob(os.path.join(PMASK_IMGS, "*.png")))
ok, miss = 0, 0
for img_path in all_png:
    base = os.path.splitext(os.path.basename(img_path))[0]
    found = find_label_path(base)
    if found is None:
        miss += 1
        continue
    _ldir, _stem, lbl_path = found
    bgr = cv2.imread(img_path)
    if bgr is None:
        miss += 1
        continue
    h0, w0 = bgr.shape[:2]
    full_lbl = build_multiclass_from_labels(lbl_path, h0, w0)
    hsv_strawberry_refine(bgr, full_lbl)  # fill missing fruit in red channel
    # ensure peduncle still on top where annotated
    for line in open(lbl_path).read().strip().splitlines():
        m = yolo_line_to_full_mask(h0, w0, line, PED_CLS)
        if m is not None:
            full_lbl[m > 0] = 2
    boxes = yolo_all_boxes(img_path)
    if not boxes:  # fallback: use full image if YOLO finds nothing
        boxes = [(0, 0, w0, h0)]
    skipped_bg = 0
    for idx, (x1, y1, x2, y2) in enumerate(boxes):
        crop = bgr[y1:y2, x1:x2].copy()
        mcp  = full_lbl[y1:y2, x1:x2].copy()
        if np.count_nonzero(mcp) < MIN_CONTENT:  # skip pure-background crops
            skipped_bg += 1
            continue
        outn = f"{base}_{idx:02d}.png"
        cv2.imwrite(os.path.join(OUT_IMGS, outn), crop)
        cv2.imwrite(os.path.join(OUT_LBL, outn), mcp)
        ok += 1
print(f"Saved {ok} per-fruit crops with 3-class masks across {len(all_png) - miss} source images.")
print(f"Skipped {miss} (no label) + {skipped_bg} (pure-background crops).")

## Cell 4 — Visualise 3-class label maps (random sample)

Reads 6 randomly selected masks from `multiclass_masks/masks_3class/` and displays them with a colour key:
- **Black** = class 0 (background)
- **Blue** = class 1 (strawberry fruit body)
- **Green** = class 2 (peduncle / crown)

Saves the figure to `multiclass_masks/sample_multiclass.png`.

In [ ]:
import matplotlib.pyplot as plt
cmap = np.zeros((256, 3), dtype=np.uint8)
cmap[0] = [0,0,0]; cmap[1] = [0,0,255]; cmap[2] = [0,255,0]
import random
all_masks = glob.glob(os.path.join(OUT_LBL, "*.png"))
samp = random.sample(all_masks, min(6, len(all_masks)))
if samp:
    fig, axs = plt.subplots(2, 3, figsize=(12, 6))
    for ax, p in zip(axs.ravel(), samp):
        m = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        if m is None:
            continue
        if m.ndim == 3:
            m = m.squeeze()  # cv2 may return (H,W,1) for single-channel PNGs
        m = np.clip(m.astype(np.int32), 0, 2).astype(np.uint8)
        vis = cmap[m]
        ax.imshow(vis)
        ax.set_title(os.path.basename(p)[:30], fontsize=7)
        ax.axis("off")
    plt.suptitle("3-class label maps: black=0 bg, blue=1 fruit, green=2 peduncle", fontsize=10)
    plt.tight_layout()
    pfig = os.path.join(PROJECT, "multiclass_masks", "sample_multiclass.png")
    plt.savefig(pfig, dpi=120)
    plt.show()
    print("Saved sample:", pfig)